<a href="https://colab.research.google.com/github/Innocent-Gershon/accra-flood-risk-prediction/blob/main/Innocent_Nangah's_Initial_software_product_solution_demonstration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install imbalanced-learn -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/flood-project/data/raw/gha-rainfall-subnat-full.csv"

Mounted at /content/drive


In [3]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.info()
display(df.head())
print(df.isna().sum())

(369225, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 369225 entries, 0 to 369224
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   date       369225 non-null  object 
 1   adm_level  369225 non-null  int64  
 2   adm_id     369225 non-null  int64  
 3   PCODE      369225 non-null  object 
 4   n_pixels   369225 non-null  float64
 5   rfh        369225 non-null  float64
 6   rfh_avg    369225 non-null  float64
 7   r1h        368775 non-null  float64
 8   r1h_avg    368775 non-null  float64
 9   r3h        367425 non-null  float64
 10  r3h_avg    367425 non-null  float64
 11  rfq        369225 non-null  float64
 12  r1q        368775 non-null  float64
 13  r3q        367425 non-null  float64
 14  version    369225 non-null  object 
dtypes: float64(10), int64(2), object(3)
memory usage: 42.3+ MB


,date,adm_level,adm_id,PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,1981-01-01,1,900854,GH02,798.0,1.269424,3.642774,NaN,NaN,NaN,NaN,72.539480,NaN,NaN,final
1,1981-01-11,1,900854,GH02,798.0,1.101504,2.760902,NaN,NaN,NaN,NaN,78.618484,NaN,NaN,final
2,1981-01-21,1,900854,GH02,798.0,2.075188,6.756349,4.446116,13.160025,NaN,NaN,60.181843,52.015984,NaN,final
3,1981-02-01,1,900854,GH02,798.0,4.845865,12.038555,8.022556,21.555805,NaN,NaN,57.785797,49.038456,NaN,final
4,1981-02-11,1,900854,GH02,798.0,22.137844,19.411488,29.058897,38.206390,NaN,NaN,111.168335,78.828384,NaN,final


date            0
adm_level       0
adm_id          0
PCODE           0
n_pixels        0
rfh             0
rfh_avg         0
r1h           450
r1h_avg       450
r3h          1800
r3h_avg      1800
rfq             0
r1q           450
r3q          1800
version         0
dtype: int64


In [4]:
df["date"] = pd.to_datetime(df["date"])

print("Date range:", df["date"].min(), "to", df["date"].max())
print(df.groupby("adm_level")["PCODE"].nunique())   # units per admin level
print(df["version"].value_counts())

# are the NaNs only at the start of each unit's series?
early = df.sort_values("date").groupby("adm_id").head(9)
print(df.drop(early.index)[["r1h", "r3h"]].isna().sum())

Date range: 1981-01-01 00:00:00 to 2026-07-21 00:00:00
adm_level
1     10
2    215
Name: PCODE, dtype: int64
version
final     368550
prelim       675
Name: count, dtype: int64
r1h    0
r3h    0
dtype: int64


In [5]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/flood-project/data/raw/gha_admin_boundaries.xlsx"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
print(os.listdir("/content/drive/MyDrive/flood-project/data/raw/"))

['gha-rainfall-subnat-full.csv', 'gha_admin_boundaries.xlsx']


In [7]:
boundaries = pd.read_excel("/content/drive/MyDrive/flood-project/data/raw/gha_admin_boundaries.xlsx", sheet_name=None)
print(boundaries.keys())

dict_keys(['gha_admin0', 'gha_admin0_em', 'gha_admin1', 'gha_admin1_em', 'gha_admin2', 'gha_admin2_em', 'gha_admincapitals', 'gha_adminlines', 'gha_adminlines_em', 'gha_adminpoints'])


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/flood-project/data/raw/gha-rainfall-subnat-full.csv")
print(df.shape)

(369225, 15)


In [12]:
df["date"] = pd.to_datetime(df["date"])
df_accra = df[
    (df["adm_level"] == 1) &
    (df["PCODE"] == "GH07") &
    (df["version"] == "final")
].copy()
df_accra = df_accra.sort_values("date").reset_index(drop=True)
print(df_accra.shape)

(1638, 15)


In [13]:
import numpy as np

df_accra["month"] = df_accra["date"].dt.month
monthly_threshold = df_accra.groupby("month")["rfh"].transform(lambda x: x.quantile(0.90))
df_accra["is_heavy_rain"] = (df_accra["rfh"] > monthly_threshold).astype(int)

for lag in [1, 2, 3]:
    df_accra[f"rfh_lag{lag}"] = df_accra["rfh"].shift(lag)
    df_accra[f"rfq_lag{lag}"] = df_accra["rfq"].shift(lag)
df_accra["r1h_lag1"] = df_accra["r1h"].shift(1)
df_accra["r3h_lag1"] = df_accra["r3h"].shift(1)

df_accra["rfh_roll6_mean"] = df_accra["rfh"].shift(1).rolling(6).mean()
df_accra["rfh_roll6_std"] = df_accra["rfh"].shift(1).rolling(6).std()
df_accra["month_sin"] = np.sin(2 * np.pi * df_accra["month"] / 12)
df_accra["month_cos"] = np.cos(2 * np.pi * df_accra["month"] / 12)

df_accra["target_next_heavy"] = df_accra["is_heavy_rain"].shift(-1)

feature_cols = [
    "rfh_lag1", "rfh_lag2", "rfh_lag3",
    "rfq_lag1", "rfq_lag2", "rfq_lag3",
    "r1h_lag1", "r3h_lag1",
    "rfh_roll6_mean", "rfh_roll6_std",
    "month_sin", "month_cos"
]

df_model = df_accra.dropna(subset=["target_next_heavy"] + feature_cols).copy()
print("Model-ready shape:", df_model.shape)

Model-ready shape: (1628, 30)


In [20]:
import os
os.makedirs("/content/drive/MyDrive/flood-project/data/processed", exist_ok=True)

export_cols = ["date"] + feature_cols
df_model[export_cols].to_csv(
    "/content/drive/MyDrive/flood-project/data/processed/model_features.csv",
    index=False
)
print("Exported.")

Exported.


In [14]:
X = df_model[feature_cols]
y = df_model["target_next_heavy"]
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
print("Train:", X_train.shape, "Positive rate:", y_train.mean())
print("Test:", X_test.shape, "Positive rate:", y_test.mean())

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print("After SMOTE:", X_train_balanced.shape, "Positive rate:", y_train_balanced.mean())

Train: (1302, 12) Positive rate: 0.09831029185867896
Test: (326, 12) Positive rate: 0.1196319018404908
After SMOTE: (2348, 12) Positive rate: 0.5


In [15]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train_balanced, y_train_balanced)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric="logloss")
xgb.fit(X_train_balanced, y_train_balanced)
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

for name, pred, proba in [("Random Forest", rf_pred, rf_proba), ("XGBoost", xgb_pred, xgb_proba)]:
    print(f"\n--- {name} ---")
    print("Precision:", round(precision_score(y_test, pred), 3))
    print("Recall:", round(recall_score(y_test, pred), 3))
    print("F1:", round(f1_score(y_test, pred), 3))
    print("AUC:", round(roc_auc_score(y_test, proba), 3))


--- Random Forest ---
Precision: 0.104
Recall: 0.128
F1: 0.115
AUC: 0.502

--- XGBoost ---
Precision: 0.062
Recall: 0.051
F1: 0.056
AUC: 0.513


In [16]:
print("""
Model Comparison Summary:
                    Precision  Recall  F1     AUC
RF (8 features)       0.141    0.333  0.198  0.546
XGB (8 features)      0.125    0.205  0.155  0.566
RF (12 features)      0.104    0.128  0.115  0.502
XGB (12 features)     0.062    0.051  0.056  0.513

Finding: Adding rolling-average and cyclical month features did not improve
predictive performance, suggesting the limiting factor is the information
content of rainfall-only data at 10-day-ahead, region-level granularity,
rather than feature engineering. This is consistent with known limitations
of satellite rainfall products for extreme event detection (see Ch. 2).
""")


Model Comparison Summary:
                    Precision  Recall  F1     AUC
RF (8 features)       0.141    0.333  0.198  0.546
XGB (8 features)      0.125    0.205  0.155  0.566
RF (12 features)      0.104    0.128  0.115  0.502
XGB (12 features)     0.062    0.051  0.056  0.513

Finding: Adding rolling-average and cyclical month features did not improve
predictive performance, suggesting the limiting factor is the information
content of rainfall-only data at 10-day-ahead, region-level granularity,
rather than feature engineering. This is consistent with known limitations
of satellite rainfall products for extreme event detection (see Ch. 2).



In [18]:
import os
os.makedirs("/content/drive/MyDrive/flood-project/models", exist_ok=True)

import joblib
joblib.dump(rf, "/content/drive/MyDrive/flood-project/models/flood_model_v1.pkl")
print("Model saved.")

Model saved.
